# LangChain Practice Exercises

Complete each exercise in order. You can refer back to notebooks `01` through `10` for examples of prompt templates, chat models, chains, retrieval workflows, and more. Use your own API keys via environment variables (see earlier notebooks) and feel free to add extra cells beneath each exercise while you work.

This notebook calls language models through [OpenRouter](https://openrouter.ai). Put your `OPENROUTER_API_KEY` (and optionally `MODEL`) in a `.env` file, or the notebook will ask you for the key.

If you do not set `MODEL`, the setup cell uses a default model. Set `MODEL` in your `.env` file to try a different OpenRouter model.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")


## Exercise 1 – Prompt Template Warm‑Up
Using the ideas from `01_Prompting_with_LangChain.ipynb`, create a `PromptTemplate` that turns a topic into a short public‑service announcement. Format the final string and print it so you can inspect the result before sending it to any model.

A **prompt template** is a sentence with blanks in it, such as `"Write a short announcement about {topic}"`. The code cell already sets a `topic` variable for you. Fill the blanks, print the result, and read it before you go on. A model can only work with the words you give it, so checking the prompt first saves time later.

In [ ]:
# TODO: Build and format your PromptTemplate here
from langchain_core.prompts import PromptTemplate

# example placeholder variables
topic = "community digital safety"

raise NotImplementedError("Create a prompt template and format it with your variables.")


## Exercise 2 – LCEL Chain with ChatOpenRouter
Following the workflows in `03_Model_I_O.ipynb` and `08_Principles_for_Prompting.ipynb`, build an LCEL pipeline that:
1. Accepts a user topic.
2. Uses a `ChatPromptTemplate` plus a system instruction for tone.
3. Pipes the prompt to `ChatOpenRouter` and parses the response with `StrOutputParser`.
Print the generated text.

**LCEL** (LangChain Expression Language) joins steps with the `|` symbol. Read `prompt | model | parser` as: fill in the prompt, send it to the model, then turn the model's reply into plain text. The code cell imports the three parts you need. A **system instruction** is a hidden message that sets the model's role or tone before the user's message arrives.

In [ ]:
# TODO: Implement the LCEL pipeline (prompt | model | parser)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser

raise NotImplementedError("Compose the runnable chain and invoke it with a topic.")


## Exercise 3 – Routing with `RunnableBranch`
Using ideas from `05_Chains.ipynb` and `06_Agents.ipynb`, build a small router that dispatches a question to one of two prompts:
- If the topic contains the word "history", use a prompt that answers as a historian.
- Otherwise, use a general helper prompt.
- The router's input is a dict such as `{"topic": "..."}`, and the branch condition checks the `topic` value.
- Pass the general helper prompt as the last item to `RunnableBranch`, so every question gets a prompt, including non-history ones.

Use `RunnableBranch` to select the prompt, feed it through `ChatOpenRouter`, and print the model response. Test your router with two questions: one that contains "history" and one that does not. Check that each question takes the path you expect.

A **router** chooses a path based on its input. `RunnableBranch` checks each condition in order and uses the first prompt whose condition is true. The last item has no condition, so it acts as the default.

In [ ]:
# TODO: Build the RunnableBranch router and invoke it
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser

# Hint: the router's input is a dict such as {"topic": "..."}; the branch condition reads the "topic" key.
# Hint: use ChatOpenRouter(model=MODEL, temperature=0) for repeatable answers.
# Hint: the last item passed to RunnableBranch is the default branch.

raise NotImplementedError("Create the branch, attach the model, and test with two questions.")


## Exercise 4 – Retrieval on `kdot.txt`
Revisit `04_Retrieval.ipynb` (and the later notebooks on memory/prompt management) to build a minimal RAG flow backed by `../../data/kdot.txt`. **RAG** (retrieval-augmented generation) means: find the most relevant text first, then give that text to the model when it answers.
1. Load the lyrics as a LangChain `Document` with a plain file read (UTF‑8 encoding).
2. Split them with `RecursiveCharacterTextSplitter`.
3. Create embeddings with `OpenAIEmbeddings` (pointed at OpenRouter) and store them in an in-memory vector store (`InMemoryVectorStore`).
4. Ask a question about a recurring theme in Kendrick Lamar’s writing. Use a `PromptTemplate` that places the retrieved text into the prompt, send it to `llm`, parse the reply with `StrOutputParser`, and print the answer.
Feel free to reuse helper code from prior notebooks, but keep everything self‑contained in this cell.

**Key terms:** an **embedding** is a list of numbers that represents the meaning of a piece of text. A **vector store** keeps these lists and finds the chunks closest in meaning to a question. The setup cell creates `embeddings` and `llm` for you. It also sets `check_embedding_ctx_length=False`, which sends plain text instead of token numbers. This is a common setting for OpenAI-compatible services.

> **Note on FAISS:** `04_Retrieval.ipynb` and the `apps/kdotChat` app use **FAISS**, a library for fast similarity search. This exercise uses `InMemoryVectorStore` from `langchain_core` instead. It needs no extra install, because `langchain_core` is already part of the course setup. The trade-off is that it compares the question against every chunk. That is fine for a short text like this, but it will slow down on large collections. It also keeps everything in memory, so the store is lost when the notebook restarts. Read the [InMemoryVectorStore API reference](https://python.langchain.com/api_reference/core/vectorstores/langchain_core.vectorstores.in_memory.InMemoryVectorStore.html) for the full details.

**Think critically:** the model answers from the chunks the search returns. If the right chunk is not retrieved, the model may guess or answer wrongly, so check the retrieved text yourself.

In [ ]:
# TODO: Implement the retrieval flow over ../../data/kdot.txt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Embeddings and the chat model both go through OpenRouter
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)
llm = ChatOpenRouter(model=MODEL)

raise NotImplementedError("Load the lyrics, build the vector store, and answer a question.")
